# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20 10
2. Алматы      NaN
Астана      NaN
Шымкент    3.0
dtype: float64
3. (2, 2)
4. Series DataFrame
5. [1, 2] 5


In [15]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [17]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [18]:
# 1
print("Строк и столбцов:", df.shape)
print()
print("Типы столбцов:")
print(df.dtypes)
print()
print("Пропусков в delivery_days:", df["delivery_days"].isna().sum())
print()
for col in ["city", "category", "status", "customer_id"]:
    print(f"Разных значений в {col}:", df[col].nunique())
print()


print("Написания городов:")
print(df["city"].value_counts())
print()


test = df["customer_id"] == "TEST"
print("Тестовых заказов:", test.sum(), "| их сумма:", df.loc[test, "amount"].sum())
print("Доля TEST в общей сумме заказов:", round(df.loc[test, "amount"].sum() / df["amount"].sum() * 100, 1), "%")
print()

n_dup = len(df) - df["order_id"].nunique()
print("Строк:", len(df), "| уникальных order_id:", df["order_id"].nunique(), "| лишних строк:", n_dup)


Строк и столбцов: (5463, 8)

Типы столбцов:
order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object

Пропусков в delivery_days: 160

Разных значений в city: 14
Разных значений в category: 5
Разных значений в status: 2
Разных значений в customer_id: 2003

Написания городов:
city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64

Тестовых заказов: 3 | их сумма: 29999970.0
Доля TEST в общей сумме заказов: 27.3 %

Строк: 5463 | уникальных order_id: 5283 | лишних строк: 180


**Вывод для менеджера:** 5 463 заказа, 8 столбцов, 160 пропусков в сроке доставки. 5 категорий, 2 статуса, 2 003 клиента, 14 написаний городов (реальных городов 6).

**Проблемы для отчёта о выручке:**
1. **Города записаны по-разному** («Алматы», «алматы», «Almaty»…). Фильтр по одному названию теряет около 12% заказов города, и выручка занижена. Нужно привести названия к одному виду.
2. **3 тестовых заказа (TEST) почти на 30 млн ₸**, это 27% всей суммы. Без исключения выручка завышена больше чем на треть.
3. **180 дублей строк** (5 463 строки, но 5 283 уникальных номера заказа). Выручка завышена. Дубли нужно удалить.


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [19]:
# 2
good = df.loc[paid & not_test, "amount"]
print("Оплаченных заказов без TEST:", len(good))
print("Среднее:", round(good.mean()))
print("Медиана:", good.median())
print("25-й перцентиль:", good.quantile(0.25))
print("75-й перцентиль:", good.quantile(0.75))
print("Среднее выше медианы на", round((good.mean() / good.median() - 1) * 100), "%")


Оплаченных заказов без TEST: 4901
Среднее: 14672
Медиана: 10300.0
25-й перцентиль: 5880.0
75-й перцентиль: 18690.0
Среднее выше медианы на 42 %


**Вывод для менеджера:** типичный чек **около 10 300 ₸** (медиана), половина заказов от 5 900 до 18 700 ₸. Среднее (14 700 ₸) на 42% выше, его тянут вверх редкие дорогие заказы. В отчёте честнее показывать **медиану**: она описывает обычного покупателя.


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [20]:
# 3
cancel = df["status"] == "отменён"
AL = ["Алматы", "алматы", "АЛМАТЫ", "Almaty", " Алматы"]
sh = df["city"].isin(SH)
al = df["city"].isin(AL)

# Среднее от маски True/False = доля True
share_all = cancel.mean()
share_sh = cancel[sh].mean()
share_al = cancel[al].mean()
print("Доля отмен, все заказы:", round(share_all * 100, 1), "%")
print("Доля отмен, Шымкент:", round(share_sh * 100, 1), "%", "| заказов:", sh.sum())
print("Доля отмен, Алматы:", round(share_al * 100, 1), "%", "| заказов:", al.sum())
print("Шымкент / Алматы:", round(share_sh / share_al, 1), "раза")
print("Шымкент / все заказы:", round(share_sh / share_all, 1), "раза")


Доля отмен, все заказы: 10.2 %
Доля отмен, Шымкент: 20.4 % | заказов: 804
Доля отмен, Алматы: 7.9 % | заказов: 2188
Шымкент / Алматы: 2.6 раза
Шымкент / все заказы: 2.0 раза


**Вывод для менеджера:** отменяется 10,2% всех заказов. **В Шымкенте отмен 20,4%, это в 2,6 раза больше, чем в Алматы (7,9%)**, и вдвое выше среднего. Каждый пятый заказ в Шымкенте отменяется, причину (доставка, оплата, наличие товара) нужно разбирать в первую очередь.


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [21]:
# 4
base = paid & not_test   # оплаченные без TEST; sh и al (все написания) из задания 3

sh_sep = df.loc[base & sh & sep, "amount"].sum()
sh_oct = df.loc[base & sh & octo, "amount"].sum()
al_sep = df.loc[base & al & sep, "amount"].sum()
al_oct = df.loc[base & al & octo, "amount"].sum()

sh_change = (sh_oct - sh_sep) / sh_sep * 100
al_change = (al_oct - al_sep) / al_sep * 100
print(f"Шымкент: сентябрь {sh_sep:,.0f} ₸ → октябрь {sh_oct:,.0f} ₸, изменение {sh_change:.1f} %")
print(f"Алматы:  сентябрь {al_sep:,.0f} ₸ → октябрь {al_oct:,.0f} ₸, изменение {al_change:.1f} %")
print("Падение в Шымкенте сильнее в", round(sh_change / al_change, 1), "раза")


Шымкент: сентябрь 4,949,800 ₸ → октябрь 4,044,010 ₸, изменение -18.3 %
Алматы:  сентябрь 15,523,940 ₸ → октябрь 14,643,020 ₸, изменение -5.7 %
Падение в Шымкенте сильнее в 3.2 раза


**Вывод для менеджера:** в октябре выручка упала в обоих городах. В **Шымкенте на 18,3%** (4,95 → 4,04 млн ₸), в Алматы на 5,7% (15,5 → 14,6 млн ₸). **В Шымкенте падение в 3,2 раза сильнее.** Вместе с удвоенной долей отмен (задание 3) это главный тревожный сигнал: нужно выяснить, что сломалось в Шымкенте в октябре. Цифры пока с дублями, точные значения будут после очистки.


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [22]:
# 5
# Ошибка 1 (роняет код): & выполняется раньше, чем ==, поэтому Python сначала считает
#   "Шымкент" & df["status"] → TypeError. Каждое условие нужно взять в скобки.
# Ошибка 2 (тихо врёт): == "Шымкент" ловит только одно написание,
#   а заказы «шымкент» и «Shymkent» теряются. Нужен isin(SH).
cancel_sh = df[df["city"].isin(SH) & (df["status"] == "отменён")]
print("Отменённых заказов в Шымкенте (все написания):", len(cancel_sh))

# Для сравнения: сколько дал бы код только с одной исправленной ошибкой (со скобками)
only_one = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
print("Только написание 'Шымкент':", len(only_one), "| потеряно:", len(cancel_sh) - len(only_one))


Отменённых заказов в Шымкенте (все написания): 164
Только написание 'Шымкент': 153 | потеряно: 11


**Вывод для менеджера:** в Шымкенте **164 отменённых заказа**. Код AI падал с ошибкой из-за скобок, а после их исправления показал бы 153: он видел только написание «Шымкент» и терял 11 отмен (7%).


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [23]:
# 6
print(df["customer_id"].value_counts().head(5))
top = df["customer_id"].value_counts().index[0]
print()
print("Самый частый клиент:", top)

top_orders = df.loc[df["customer_id"] == top, ["order_id", "order_date", "city", "amount", "status"]]
print(top_orders)
print()

n_rows = len(top_orders)
n_unique = top_orders["order_id"].nunique()
print("Строк у клиента:", n_rows, "| уникальных order_id:", n_unique, "| повторов:", n_rows - n_unique)


customer_id
C01818    11
C01076     9
C01947     9
C01058     9
C00420     9
Name: count, dtype: int64

Самый частый клиент: C01818
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен

Строк у клиента: 11 | уникальных order_id: 10 | повторов: 1


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [24]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [25]:
# 8